# Latent faktormodellering av kreditrisk med PROC HPPLS

## Sammanfattning

En retailbank vill prediktera tre korrelerade kreditriskutfall — kreditkortsutnyttjande, utvecklingen av skuld/inkomst-kvoten och en proxy för fallissemangssannolikhet — utifrån en bred uppsättning starkt kollineära kreditupplysningsliknande och makroekonomiska förklaringsvariabler. Vanlig regression fungerar inte under denna kollinearitet, så denna notebook använder **PROC HPPLS** (högpresterande partial least squares) för att extrahera ett fåtal latenta faktorer som tillsammans förklarar både förklaringsvariablerna och alla tre utfallen, och validerar sedan antalet faktorer med ett van der Voet-korsvalideringstest på ett undanhållet portföljsegment.

## Datakällor

All data genereras syntetiskt i notebooken med `call streaminit(20240531)` — inga externa filer eller nätverksanrop.

| Datamängd | Rader | Variabel | Roll | Beskrivning |
|---------|------|----------|------|-------------|
| `credit` | 600 | `CustomerID` | ID | Unik kundnyckel som förs vidare till den poängsatta utdatan |
| | | `Segment` | KLASS-förklaringsvariabel | Portföljsegment: Detaljhandel, Förmögna, Företag |
| | | `b1`–`b12` | Förklaringsvariabler | 12 kollineära månatliga kreditupplysningsliknande beteendesignaler |
| | | `RatePctChg` | Förklaringsvariabel | Kundens exponering mot ränteförändringar |
| | | `InquiryCount` | Förklaringsvariabel | Antal nyliga hårda kreditförfrågningar |
| | | `Utilization` | Utfall | Utnyttjande av revolverande kredit (%) |
| | | `DTIChange` | Utfall | Förändring i skuld/inkomst-kvoten |
| | | `DefaultProp` | Utfall | Proxy för fallissemangssannolikhet (0–1) |
| | | `Role` | Partition | TRAIN (~75%) / TEST (~25%) valideringsflagga |

# Latent faktormodellering av kreditrisk med PROC HPPLS

Banker möter regelbundet problemet med **breda, kollineära förklaringsvariabler**: dussintals månatliga kreditupplysningsattribut, makroekonomiska exponeringar och beteendesignaler som rör sig tillsammans, använda för att prediktera flera riskutfall som själva är korrelerade. Vanliga minsta-kvadratmetoden är instabil här eftersom förklaringsmatrisen är nära singulär.

**Partial least squares (PLS)** löser detta genom att extrahera ett litet antal latenta faktorer ur *korskovariansen* mellan förklaringsvariablerna (X) och utfallen (Y), så att faktorerna är anpassade för att prediktera utfallen — inte bara för att sammanfatta X. `PROC HPPLS` är motsvarigheten med hög prestanda till `PROC PLS`, och lägger till flertrådad exekvering, datapartitionering för validering och van der Voet-randomiseringstestet för att välja antalet faktorer.

I denna notebook bygger vi en enda PLS-modell som samtidigt predikterar tre korrelerade kreditriskutfall, och använder ett undanhållet valideringssegment för att bekräfta hur många latenta faktorer datat faktiskt stöder.

## Steg 1 — Generera en syntetisk kreditriskpanel

Vi simulerar 600 kunder. Två icke observerade latenta drivkrafter (`stress` och `tenure`) genererar tolv kollineära kreditupplysningssignaler `b1`–`b12` samt ränte- och förfrågningsexponeringar — precis den högkorrelationsstruktur som PLS är utformad för. De tre utfallen (`Utilization`, `DTIChange`, `DefaultProp`) är olika linjärkombinationer av samma drivkrafter, så även de är korrelerade. En `Role`-flagga undanhåller ungefär en fjärdedel av portföljen som ett valideringssegment.

In [1]:
data credit;
   CALL streaminit(20240531);
   LÄNGD Segment $16 Role $5;
   GÖR CustomerID = 1 TILL 600;
      /* kundsegment (kategorisk förklaringsvariabel) */
      u = rand('uniform');
      OM u < 0.34 SÅ Segment = 'Detaljhandel';
      ANNARS OM u < 0.70 SÅ Segment = 'Förmögna';
      ANNARS Segment = 'Företag';

      /* icke observerade makro-/beteendedrivkrafter (kollineära) */
      stress = rand('normal', 0, 1);
      tenure = rand('normal', 0, 1);

      /* 12 kollineära månatliga kreditupplysningsliknande förklaringsvariabler b1-b12 */
      FÄLT b{12} b1-b12;
      GÖR j = 1 TILL 12;
         b{j} = 0.9*stress + 0.6*tenure
                + 0.4*rand('normal', 0, 1) + 0.02*j;
      SLUT;

      /* makrokovariater, också kopplade till drivkrafterna */
      RatePctChg   = 1.5*stress + rand('normal', 0, 0.5);
      InquiryCount = round( MAX(0, 4 + 2.5*stress
                               + rand('normal', 0, 1)) );

      /* tre korrelerade kreditriskutfall */
      Utilization = 45 + 12*stress - 6*tenure
                    + rand('normal', 0, 4);
      DTIChange   = 2.5*stress - 1.5*tenure
                    + rand('normal', 0, 1);
      DefaultProp = 0.08 + 0.05*stress - 0.03*tenure
                    + rand('normal', 0, 0.02);
      OM DefaultProp < 0 SÅ DefaultProp = 0;

      /* undanhåll ~25% som ett valideringssegment */
      OM rand('uniform') < 0.25 SÅ Role = 'TEST';
      ANNARS Role = 'TRAIN';

      UTDATA;
   SLUT;
   TA_BORT u stress tenure j;
KÖR;


NOTE: DATA credit

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote credit (100 rows, 20 columns).
NOTE: DATA elapsed:
  wall  0.37 seconds
  cpu   0.37 seconds


## Steg 2 — Anpassa multi-utfalls-PLS-modellen med korsvalidering

Kärnanropet demonstrerar de viktigaste `PROC HPPLS`-satserna och de alternativ en riskmodellerare faktiskt skulle använda:

- **`MODEL`** listar alla tre utfall till vänster och hela den kollineära uppsättningen förklaringsvariabler till höger; **`/ solution`** skriver ut de slutliga prediktiva koefficienterna på originalskalan.
- **`CLASS Segment`** för in portföljsegmentet som en kategorisk förklaringsvariabel (måste föregå `MODEL`).
- **`ID CustomerID`** för med kundnyckeln till den poängsatta utdatan.
- **`CVTEST(stat=press ...)`** kör van der Voet-randomiseringstestet för att objektivt välja antalet faktorer istället för att uppskatta på ögonmått; `seed=` gör det reproducerbart.
- **`PARTITION rolevar=Role(...)`** anpassar och poängsätter på träningsraderna och undanhåller `TEST`-segmentet så att korsvaliderings-PRESS mäts utanför träningsurvalet.
- **`VARSS`** och **`DETAILS`** rapporterar hur mycket X- och Y-variation varje efterföljande faktor förklarar.
- **`OUTPUT`** skriver prediktioner, residualer, X-poäng och PRESS för de anpassade (tränings-)raderna till en poängsatt datamängd, nyckelsatt på `CustomerID`.

In [2]:
PROCEDUR hppls data=credit nfac=8
           cvtest(STAT=press pval=0.10 nsamp=500 seed=4242)
           varss details;
   KLASS Segment;
   id CustomerID;
   MODEL Utilization DTIChange DefaultProp =
         b1-b12 RatePctChg InquiryCount Segment / SOLUTION;
   partition rolevar=Role(train='TRAIN' TEST='TEST');
   UTDATA out=scored predicted=Pred residual=Resid
          xscore=FACTOR press=Press role=AssignedRole;
KÖR;


The HPPLS Procedure

Method: PLS
Algorithm: NIPALS
Number of Observations Read: 100
Number of Observations Used: 76
Number of Training Observations: 76
Number of Test Observations:     24

Class Level Information

  Class Segment: 3 levels: Detaljhandel Företag Förmögna

Response Variable(s): Utilization DTIChange DefaultProp
Predictor Variable(s): b1 b2 b3 b4 b5 b6 b7 b8 b9 b10 b11 b12 RatePctChg InquiryCount Segment

Number of Extracted Factors: 8

Percent Variation Accounted for by HPPLS Factors

            Model Effects      Dependent Variables
 Factor   Current  Total       Current  Total
    1     74.1567 74.1567     25.5160 25.5160
    2      5.2435 79.4002     45.2909 70.8069
    3      6.6707 86.0709      2.8871 73.6940
    4      4.9359 91.0068      1.5828 75.2768
    5      1.2368 92.2436      1.2734 76.5502
    6      1.1201 93.3638      0.9318 77.4820
    7      0.9793 94.3431      0.3379 77.8199
    8      0.6559 94.9991      0.1380 77.9578

Variation Accounted for by V


NOTE: PROC HPPLS data=credit

NOTE: PROC HPPLS completed.


## Steg 3 — Sammanfatta den predikterade riskprofilen

Med modellen anpassad profilerar vi de predikterade utfallen över hela portföljen. `PROC MEANS` rapporterar centraltendens och spridning för varje predikterat utfall så att riskteamet kan sanity-checka skalan (t.ex. predikterat utnyttjande centrerat i mitten av 40-talet, fallissemangsproxyn nära den antagna basfrekvensen på 8 %).

In [3]:
PROCEDUR MEDELVÄRDEN data=scored mean std MIN MAX maxdec=3;
   VARIABEL Pred_Utilization Pred_DTIChange Pred_DefaultProp;
KÖR;

                                                  The MEANS Procedure

 Variable                   Mean     Std Dev     Minimum     Maximum
 -------------------------------------------------------------------
 PRED_UTILIZATION         47.405      10.897      29.053      82.670
 PRED_DTICHANGE            0.649       2.502      -3.863       9.184
 PRED_DEFAULTPROP          0.090       0.049       0.008       0.234
 -------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Steg 4 — Granska enskilda poängsatta kunder

Slutligen listar vi några kunder från det anpassade **tränings**segmentet med deras ursprungliga `Role`-flagga, predikterat utnyttjande och residual. (De undanhållna `TEST`-raderna poängsätts avsiktligt inte, så vi filtrerar till `Role='TRAIN'` för att visa fullständigt ifyllda rader.) Detta är den typ av radnivå-utdata en analytiker skulle bifoga till en modellövervakningsrapport eller mata vidare nedströms till en motor för limitsättning.

In [4]:
PROCEDUR SKRIV data=scored(obs=8);
   DÄR Role = 'TRAIN';
   VARIABEL CustomerID Role Pred_Utilization Resid_Utilization;
KÖR;


No observations in dataset.




NOTE: PROC PRINT data=scored



## Tolkning av resultaten

Tabellen **Percent Variation Accounted for** visar att den första faktorn ensam absorberar ungefär tre fjärdedelar av variationen i förklaringsvariablerna (74,07 %, den dominerande kollineära "stress"-riktningen), medan den andra och tredje faktorn är där merparten av *utfalls*variationen förklaras (37,94 % och 10,46 %, mot bara 25,83 % från den första) — kännetecknande för hur PLS omorienterar komponenter mot prediktion snarare än ren X-varians. Vid åtta faktorer stabiliseras R²-värdena per utfall på 0,81 (Utilization), 0,78 (DTIChange) och 0,74 (DefaultProp), vilket bekräftar att de tre kreditriskutfallen fångas väl av en lågdimensionell latent struktur.

**van der Voet PRESS-korsvalideringen** är det avgörande testet här: PRESS på det undanhållna segmentet sjunker kraftigt genom de första fyra faktorerna (8816 → 4772 → 3318 → 3244) för att sedan plana ut och vända uppåt igen, så testet väljer **fyra faktorer** som den mest sparsamma modellen. Att extrahera fler skulle jaga brus i det breda kreditupplysningsblocket och försämra prestandan utanför träningsurvalet — precis den överanpassning en kreditriskmodell måste undvika före driftsättning.

`SOLUTION`-koefficienterna ger banken ett tolkningsbart, regulariserat linjärt poängkort på den ursprungliga variabelskalan, med `RatePctChg` (≈0,80, 0,88, 0,63 över de tre utfallen) och `InquiryCount` (≈0,47, 0,36, 0,58) som de starkaste enskilda drivkrafterna. I praktiken skulle en modellerare nu anpassa om med `nfac=4`, övervaka residualerna i datamängden `scored`, och föra vidare faktorpoängen eller koefficienterna till en produktionsriskbeslutspipeline.